# Pipeline 

Este notebook contém todas as instruções necessárias para executar a pipeline desenvolvida na Sprint 04 pelo nosso grupo.

---

## Por que um notebook?

Optamos por utilizar um **notebook Jupyter** porque ele:

- Permite **executar cada etapa da pipeline separadamente**, de forma interativa.
- Garante **reprodutibilidade**: outros usuários podem executar exatamente os mesmos passos e obter os mesmos resultados.
- Facilita o entendimento do processo, com **códigos explicativos e células comentadas**.

---

## Pré-requisitos: preciso instalar bibliotecas?

Sim. Nossa pipeline depende de bibliotecas como `ultralytics`, `torch`, `opencv-python`, entre outras.

Mas não se preocupe!

As células do notebook já trazem os comandos necessários para instalar todas as dependências automaticamente com `pip`.  
**Você só precisa executá-las**.  
> ⚠️ Caso ocorra algum erro na instalação, tente novamente **pelo terminal do sistema**.

---

## Como executar a pipeline?

Você tem três opções:

### **1. Usar o Google Colab (mais simples)**

1. Acesse: [https://colab.research.google.com](https://colab.research.google.com)
2. Faça login com sua conta Google.
3. Faça upload do notebook e dos arquivos necessários.
4. Execute célula por célula, clicando no botão ▶️ à esquerda.

> Vantagens: já vem com GPU gratuita, ambiente pronto e interface fácil. O grupo utilizou o colab em momentos fora da faculdade. 

---

### **2. Rodar localmente com Jupyter Notebook**

#### A) Usando o Visual Studio Code (VSCode)

1. Instale o [VSCode](https://code.visualstudio.com/)
2. Instale a extensão **Jupyter** no próprio VSCode
3. Instale o Python e Jupyter via terminal:

```bash
pip install notebook
pip install jupyterlab
```

4. Acesse o arquivo e selecione "Run All". 

OBS: Você também pode baixar o Jupyter Notebook pelas extensões do VsCode. 

#### B) Usando Anaconda

1. Instale o [Anaconda](https://www.anaconda.com/products/distribution)
2. Abra o **Anaconda Navigator**
3. Execute o **Jupyter Notebook** a partir do menu
4. Navegue até o diretório onde está salvo o notebook da pipeline e clique para abri-lo
5. Execute célula por célula clicando no botão de start

In [ ]:
# Clonamos o repositório do Yolo Crack, um repositório que contém um modelo de segmentação refinado para identificar rachaduras

!git clone https://huggingface.co/OpenSistemas/YOLOv8-crack-seg

In [ ]:
# Instalamos a biblioteca Ultralytics, que contém o YOLO
!pip install -q ultralytics

### À partir do repositório que clonamos, buscamos o modelo presente e fazemos as nossas máscaras de segmentação

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np

segment_model = YOLO("YOLOv8-crack-seg/yolov8m/weights/best.pt")

input_folder = "test_images/"
overlay_folder = "overlay_for_classification/"
os.makedirs(overlay_folder, exist_ok=True)

for img_file in os.listdir(input_folder):
    if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')): continue

    img_path = os.path.join(input_folder, img_file)
    results = segment_model.predict(source=img_path, save=False, conf=0.35)

    img = cv2.imread(img_path)
    overlay = img.copy()
    masks = results[0].masks

    if masks is not None:
        for m in masks.data:
            m = m.cpu().numpy()
            m_resized = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            mask_uint8 = (m_resized * 255).astype(np.uint8)
            color = (0, 0, 255)  # vermelho
            overlay[mask_uint8 > 0] = cv2.addWeighted(overlay, 0.7, np.full_like(overlay, color), 0.3, 0)[mask_uint8 > 0]

    save_path = os.path.join(overlay_folder, img_file)
    cv2.imwrite(save_path, overlay)
    print(f"Overlay salvo: {img_file}")


### Treinamos o modelo de detecção

In [ ]:
model = YOLO("yolov8m.pt")
model.train(
    data="config.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    lr0=1e-3,
    device="CPU",
    cache=True
)

### Iniciamos o treino do nosso modelo de classificação

In [ ]:
model = YOLO("yolov8m-cls.pt")
model.train(
    data="/content/dataset_dd",  #Essa era a pasta utilizada pelo grupo enquanto estávamos no colab, você deve alterar esse caminho para o certo no seu ambiente
    epochs=50,
    imgsz=224,
    batch=32,
    device="CPU"
)

### À partir dos crops feitos inicialmente pelo modelo de segmentação, conseguimos fazer as predições nas imagens 


In [ ]:
model = YOLO("runs/classify/train12/weights/best.pt") #Importante mencionar que esse caminho era o utilizado pelo grupo, você deve alterar essa linha para o caminho que corresponde ao seu modelo
input_folder = "overlay_for_classification/"
output_folder = "classified_images/"
os.makedirs(output_folder, exist_ok=True)

for img_file in os.listdir(input_folder):
    if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')): continue

    img_path = os.path.join(input_folder, img_file)
    img = cv2.imread(img_path)

    results = model.predict(source=img_path, save=False)
    label = results[0].names[results[0].probs.top1]
    score = results[0].probs.top1conf

    h, w = img.shape[:2]
    start_point = (0, 0)
    end_point = (w - 1, h - 1)

    color = (0, 255, 0) if label == "retração" else (0, 0, 255)

    cv2.rectangle(img, start_point, end_point, color, 2)
    text = f"{label} ({score:.2f})"
    cv2.putText(img, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    out_path = os.path.join(output_folder, img_file)
    cv2.imwrite(out_path, img)

    print(f"{img_file} → {text}")


### Você precisa de um dataset corretamente formatado para rodar esse código. 

#### Como deve estar a estrutura da pasta do dataset de classificação?

Para que o treinamento funcione corretamente, seu diretório deve estar organizado desta forma:


```bash
dataset_cls/
├── train/
│ ├── retracao/
│ │ ├── img1.jpg
│ │ ├── img2.jpg
│ └── termica/
│ ├── img3.jpg
│ ├── img4.jpg
├── val/
│ ├── retracao/
│ │ ├── img5.jpg
│ └── termica/
│ ├── img6.jpg
```


Cada subpasta dentro de `train/` e `val/` representa uma **classe**.  
O nome das pastas **deve coincidir** com os nomes definidos no arquivo `config_cls.yaml`:

```yaml
nc: 2
names: ['retração', 'térmica']


